In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)

In [3]:
dim_carrier = pd.read_csv("Lookups/carrier_dim.csv")
dim_port = pd.read_csv("Lookups/port.csv")
dim_hazmat_type = pd.read_csv("Lookups/hazmat_type.csv")
save_csv(dim_carrier, "Gold-Layer/dim_carrier.csv")
save_csv(dim_port, "Gold-Layer/port.csv")
save_csv(dim_hazmat_type, "Gold-Layer/hazmat_type.csv")

In [4]:
header = pd.read_csv("Lookups/header_with_ids.csv")
shipment_container = pd.read_csv("Lookups/shipment_container.csv")
hazmat_assignment = pd.read_csv("Lookups/hazmat_assignment.csv")

C:\Users\CY522SH\AppData\Local\Temp\ipykernel_31008\2470289111.py:1: DtypeWarning: Columns (0: carrier_code, 1: vessel_country_code, 2: vessel_name, 3: foreign_port_of_lading_qualifier, 4: estimated_arrival_date, 5: manifest_unit, 6: weight_unit, 7: record_status_indicator, 8: place_of_receipt, 9: foreign_port_of_destination_qualifier, 10: conveyance_id_qualifier, 11: in_bond_entry_type, 12: second_party, 13: actual_arrival_date) have mixed types. Specify dtype option on import or set low_memory=False.
  header = pd.read_csv("Lookups/header_with_ids.csv")


In [5]:
hazmat_silver = pd.read_csv("Silver-Layer/hazmat_class_silver.csv")

In [6]:

dim_date = (
    header[["actual_arrival_date"]]
    .dropna()
    .drop_duplicates()
    .rename(columns={"actual_arrival_date": "full_date"})
    .reset_index(drop=True)
)


dim_date["full_date"] = pd.to_datetime(
    dim_date["full_date"],
    errors="coerce"
)

dim_date = dim_date.dropna(subset=["full_date"])


dim_date["date_id"] = range(1, len(dim_date) + 1)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["day"] = dim_date["full_date"].dt.day


save_csv(dim_date, "Gold-Layer/dim_date.csv")

In [7]:

print(hazmat_assignment.columns.tolist())
print(dim_hazmat_type.columns.tolist())



['hazmat_assignment_id', 'container_number']
['hazmat_sequence_number', 'hazmat_classification']


In [7]:
hazmat_assign_small = hazmat_assignment[["container_number"]]
hazmat_silver_small = hazmat_silver[["container_number", "hazmat_sequence_number"]]
hazmat_type_small = dim_hazmat_type[["hazmat_sequence_number"]]

In [8]:
carrier_map = header.set_index("identifier")["carrier_code"]
port_map    = header.set_index("identifier")["unlading_port_id"]
date_map    = pd.to_datetime(
    header.set_index("identifier")["actual_arrival_date"],
    errors="coerce",
    cache=False
)
weight_map  = header.set_index("identifier")["weight"]

In [10]:
CHUNK_SIZE = 500_000

output_path = "Gold-Layer/fact_base_atomic.csv"
first_chunk = True


In [ ]:
for chunk in pd.read_csv(
    "Lookups/shipment_container.csv",
    chunksize=CHUNK_SIZE
):
    fb_chunk = (
        chunk
        .merge(hazmat_assign_small, on="container_number", how="inner")
        .merge(hazmat_silver_small, on="container_number", how="inner")
        .merge(hazmat_type_small, on="hazmat_sequence_number", how="inner")
    )

    # Map header attributes (memory-safe)
    fb_chunk["carrier_id"] = fb_chunk["identifier"].map(carrier_map)
    fb_chunk["port_id"] = fb_chunk["identifier"].map(port_map)
    fb_chunk["actual_arrival_date"] = fb_chunk["identifier"].map(date_map)
    fb_chunk["weight"] = fb_chunk["identifier"].map(weight_map)

    
    # Drop rows where required IDs did not map
    fb_chunk = fb_chunk.dropna(
        subset=["carrier_id", "port_id", "actual_arrival_date"]
    )

    # Enforce correct dtypes
    fb_chunk["carrier_id"] = fb_chunk["carrier_id"].astype("int64")
    fb_chunk["port_id"] = fb_chunk["port_id"].astype("int64")

    fb_chunk["actual_arrival_date"] = pd.to_datetime(
        fb_chunk["actual_arrival_date"],
        errors="coerce"
    )

    # Optional but safe
    fb_chunk["weight"] = fb_chunk["weight"].astype("float64")


    # PROJECT ONLY — NO DEDUPLICATION
    fb_chunk = fb_chunk[
        [
            "identifier",
            "container_number",
            "hazmat_sequence_number",
            "carrier_id",
            "port_id",
            "actual_arrival_date",
            "weight",
        ]
    ]

    fb_chunk.to_csv(
        output_path,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
    )

    first_chunk = False
    del fb_chunk